In [0]:

import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
feature_table_name = f"{current_catalog}.default.wine_features"

df_spark = spark.table(feature_table_name)
df_pandas = df_spark.toPandas()

feature_cols = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide", "density",
    "pH", "sulphates", "alcohol", "alcohol_density_ratio", "total_acidity"
]

X = df_pandas[feature_cols]
y = df_pandas["is_high_quality"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

experiment_name = "/Shared/control-2"
mlflow.set_experiment(experiment_name)

models_config = [
    {
        "name": "modelo-run-1",
        "model": RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42),
        "params": {"model_type": "RandomForest", "n_estimators": 50, "max_depth": 5}
    },
    {
        "name": "modelo-run-2",
        "model": RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42),
        "params": {"model_type": "RandomForest", "n_estimators": 150, "max_depth": 10}
    },
    {
        "name": "modelo-run-3",
        "model": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
        "params": {"model_type": "GradientBoosting", "n_estimators": 100, "learning_rate": 0.1}
    },
    {
        "name": "modelo-run-4",
        "model": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42),
        "params": {"model_type": "GradientBoosting", "n_estimators": 200, "learning_rate": 0.05}
    },
    {
        "name": "modelo-run-5",
        "model": LogisticRegression(max_iter=1000, C=1.0, random_state=42),
        "params": {"model_type": "LogisticRegression", "max_iter": 1000, "C": 1.0}
    }
]


for config in models_config:
    run_name = config["name"]
    model = config["model"]
    params = config["params"]
    
    with mlflow.start_run(run_name=run_name):
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds)
        rec = recall_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("precision", prec)
        mlflow.log_metric("recall", rec)
        mlflow.log_metric("f1_score", f1)
        
        mlflow.sklearn.log_model(model, artifact_path="model")
        
        print(f"Ejecutado {run_name} exitosamente.")

In [0]:
import mlflow
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient


mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

best_model = models_config[1]["model"]


signature = infer_signature(X_test, y_test)

with mlflow.start_run(run_name="modelo-run-2-principal") as run:
    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="model",
        signature=signature
    )
    run_id = run.info.run_id


full_model_name = "workspace.default.modelo_mlops"
model_uri = f"runs:/{run_id}/model"

registered_model = mlflow.register_model(model_uri=model_uri, name=full_model_name)

client.set_registered_model_alias(
    name=full_model_name,
    alias="PRINCIPAL",
    version=registered_model.version
)

print(f"✅ Modelo registrado correctamente: {full_model_name}")
print(f"✅ Alias 'PRINCIPAL' asignado exitosamente a la Versión {registered_model.version}")